# Model 4 — Conditional Diffusion Date Generator

Run on a **T4 GPU** Colab: Runtime → Change runtime type → T4 GPU

Architecture: **DDPM** (predict-x0) trained with MSE + auxiliary CE loss.  
Inference: **DDIM** (50 steps) — fast and deterministic.  
Constraint projection mirrors Models 1–3 (month, decade, leap-year, day-of-week).

> ⏱ 100 epochs takes ~15–25 min on T4 (much faster than the EBM version).

In [ ]:
# ── 1. Clone & install ────────────────────────────────────────────────────────
REPO = "https://github.com/SalmaSherif7070/Conditional-Date-Generation-Using-Deep-Generative-Models"
!git clone {REPO} repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# ── 2. GPU check ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NOT available — set Runtime → T4 GPU')

In [ ]:
# ── 3. Create __init__.py files ───────────────────────────────────────────────
import os
for pkg in ['src', 'src/model_1', 'src/model_2', 'src/model_3', 'src/model_4']:
    os.makedirs(pkg, exist_ok=True)
    p = os.path.join(pkg, '__init__.py')
    if not os.path.exists(p):
        open(p, 'w').close()
print('✓ __init__.py ready')

In [ ]:
# ── 4. Patch config.py — 100 epochs, flat output paths, diffusion hyperparams ─
config_src = '''
from dataclasses import dataclass

@dataclass
class ModelConfig:
    cond_dim: int = 128
    max_decade: int = 300

@dataclass
class TrainConfig:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 2e-3
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model2Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train2Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr_g: float = 1e-4
    lr_d: float = 4e-4
    n_critic: int = 2
    lambda_gp: float = 10.0
    tau_start: float = 2.0
    tau_end: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model3Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train3Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-3
    beta_max: float = 0.5
    beta_warmup_frac: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model4Config:
    # Condition / date embedding
    cond_dim: int = 128        # condition embedding dim
    max_decade: int = 300      # max decade index
    emb_dim: int = 32          # per-component date embedding dim (4x32 = 128 total)
    # Denoising network
    hidden_dim: int = 512      # residual MLP hidden width
    n_layers: int = 6          # number of FiLM residual blocks
    time_dim: int = 128        # sinusoidal time embedding dim
    # Diffusion schedule
    T: int = 500               # forward process timesteps

@dataclass
class Train4Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-4
    aux_weight: float = 0.1    # weight of auxiliary CE loss
    ddim_steps: int = 50       # DDIM inference steps
    val_split: float = 0.2
    seed: int = 42

@dataclass
class PathConfig:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path2Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    generator_path: str = "output/generator.pt"
    discriminator_path: str = "output/discriminator.pt"
    encoder_path: str = "output/encoder.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path3Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path4Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"
'''
with open('src/config.py', 'w') as f:
    f.write(config_src.strip())
print('✓ src/config.py patched (100 epochs, diffusion hyperparams)')

In [ ]:
# ── 5. Overwrite src/model_4/ with diffusion implementation ──────────────────
# Downloads each file directly from the new implementation.
# If you have the files locally, upload them to Colab and copy instead.

MODEL4_FILES = {
    'src/model_4/model.py':         'MODEL4_MODEL_PY',
    'src/model_4/train.py':         'MODEL4_TRAIN_PY',
    'src/model_4/evaluate.py':      'MODEL4_EVALUATE_PY',
    'src/model_4/predict.py':       'MODEL4_PREDICT_PY',
    'src/model_4/visualization.py': 'MODEL4_VIS_PY',
}

# ── Paste model.py ───────────────────────────────────────────────────────────
model_py = r"""
"""  # <-- PASTE src/model_4/model.py content here if not using git pull

# If your repo already has the updated files after a push, just run:
# !git pull origin main
# Otherwise upload the 5 files via Colab file browser and copy them.

print('Tip: if you pushed the new model_4 files to your repo, run:')
print('  !git pull origin main')
print('in a new cell, then skip to cell 6.')

In [ ]:
# ── 5b. Pull updated model_4 files from your repo ────────────────────────────
# Run this cell AFTER pushing the new model_4/*.py files to GitHub.
# If the files are already in the initial clone, this cell is a no-op.
!git pull origin main 2>&1 | tail -5
print('✓ Repo up-to-date')

In [ ]:
# ── 6. Create output directories ─────────────────────────────────────────────
import os
os.makedirs('output/figures', exist_ok=True)
print('✓ Directories ready')

In [ ]:
# ── 7. Quick sanity check — model forward pass ───────────────────────────────
import sys
sys.path.insert(0, '.')
import torch
from src.config import Model4Config, Train4Config
from src.model_4.model import DateDiffusionModel

cfg  = Model4Config()
tcfg = Train4Config()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = DateDiffusionModel(
    cond_dim=cfg.cond_dim, max_decade=cfg.max_decade,
    hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
    time_dim=cfg.time_dim, T=cfg.T, emb_dim=cfg.emb_dim,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'✓ Model created — {n_params:,} parameters')

# Dummy forward
X_dummy = torch.randint(0, 7, (4, 4)).to(device)
X_dummy[:, 1] = torch.randint(0, 12, (4,))
X_dummy[:, 2] = torch.randint(0, 2,  (4,))
X_dummy[:, 3] = torch.randint(0, 20, (4,))
Y_dummy = torch.tensor([[15, 6, 204], [1, 1, 200], [28, 2, 204], [10, 12, 199]], dtype=torch.long).to(device)

loss, mse, ce = model.training_loss(X_dummy, Y_dummy)
print(f'  Training loss: {loss.item():.4f}  (MSE={mse:.4f}, CE={ce:.4f})')

with torch.no_grad():
    Y_sample = model.sample(X_dummy, ddim_steps=10, device=device)
print(f'  Sample shape: {Y_sample.shape}  (expected [4, 3])')
print(f'  Sample dates: {Y_sample.tolist()}')
del model

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
!python main.py train4

In [ ]:
# ── 9. Evaluate ───────────────────────────────────────────────────────────────
!python main.py evaluate4

In [ ]:
# ── 10. Predict ───────────────────────────────────────────────────────────────
!python main.py predict4 \
    -i data/raw/example_input.txt \
    -o output/predictions.txt

print('\nFirst 10 predictions:')
with open('output/predictions.txt') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(line, end='')

In [ ]:
# ── 11. Display figures ───────────────────────────────────────────────────────
from IPython.display import Image, display
import glob
figs = sorted(glob.glob('output/figures/*.png'))
print(f'Found {len(figs)} figures:')
for path in figs:
    print('\n', path)
    display(Image(path))

In [ ]:
# ── 12. (Optional) Faster inference demo — vary DDIM steps ───────────────────
# Trade off speed vs quality. 10 steps is very fast, 100 steps is highest quality.
import torch, time
from src.config import Model4Config, Train4Config, Path4Config
from src.model_4.model import DateDiffusionModel
from src.data_processing import load_example_input

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = Model4Config(); pcfg = Path4Config()

model = DateDiffusionModel(
    cond_dim=cfg.cond_dim, max_decade=cfg.max_decade,
    hidden_dim=cfg.hidden_dim, n_layers=cfg.n_layers,
    time_dim=cfg.time_dim, T=cfg.T, emb_dim=cfg.emb_dim,
).to(device)
model.load_state_dict(torch.load(pcfg.weights_path, map_location=device))
model.eval()

X, _ = load_example_input(pcfg.example_input_path)
X = X.to(device)

for steps in [10, 25, 50, 100]:
    t0 = time.time()
    with torch.no_grad():
        Y = model.sample(X, ddim_steps=steps, device=device)
    dt = time.time() - t0
    print(f'  ddim_steps={steps:>3}  →  {dt:.2f}s for {len(X)} samples')

In [ ]:
# ── 13. Zip & download ────────────────────────────────────────────────────────
import zipfile, os
ZIP = 'output/model4_outputs.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir('output/figures')):
        zf.write(f'output/figures/{fn}', f'figures/{fn}')
    if os.path.exists('output/predictions.txt'):
        zf.write('output/predictions.txt', 'predictions.txt')
    if os.path.exists('output/weights.pt'):
        zf.write('output/weights.pt', 'weights.pt')
print(f'✓ ZIP ({os.path.getsize(ZIP)/1e6:.1f} MB) → {ZIP}')
from google.colab import files
files.download(ZIP)